# SmolLM3 staff-support evaluation

This notebook evaluates SmolLM3-3B using the same fictional reference,
questions and instructions as Qwen. Upload "qwen_batch_backup.zip". Ouput from the first notebook.

In [1]:
%pip install -q transformers accelerate safetensors

In [2]:
from google.colab import files
from pathlib import Path
import zipfile
import json
import hashlib
import torch

assert torch.cuda.is_available(), "Select a T4 GPU runtime."

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
assert len(zip_names) == 1, "Upload one Qwen backup ZIP."

project = Path("/content/staff_comparison")
project.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_names[0]) as archive:
    for name in archive.namelist():
        target = (project / name).resolve()
        assert target.is_relative_to(project.resolve()), "Invalid ZIP path."
    archive.extractall(project)

reference = (project / "data" / "reference.txt").read_text()
dataset_text = (project / "data" / "test_cases.json").read_text()
test_cases = json.loads(dataset_text)

fingerprint = hashlib.sha256(
    (reference + "\n" + dataset_text).encode()
).hexdigest()

assert fingerprint == (
    project / "data" / "dataset_sha256.txt"
).read_text().strip(), "Dataset fingerprint mismatch."

qwen_configs = sorted(
    (project / "results").glob("qwen_config_*.json")
)
assert qwen_configs, "Qwen batch configuration not found."

qwen_config = json.loads(qwen_configs[-1].read_text())

system_prompt = qwen_config["system_prompt"]
generation_settings = qwen_config["generation_settings"].copy()

assert len(test_cases) == 30

print("GPU:", torch.cuda.get_device_name(0))
print("Restored questions:", len(test_cases))
print("Dataset fingerprint verified.")
print("Generation settings:", generation_settings)

Saving qwen_batch_backup.zip to qwen_batch_backup.zip
GPU: Tesla T4
Restored questions: 30
Dataset fingerprint verified.
Generation settings: {'max_new_tokens': 160, 'do_sample': False}


## Load SmolLM3

The model is loaded in float16.

In [3]:
import time
import platform
import importlib.metadata
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import model_info

model_id = "HuggingFaceTB/SmolLM3-3B"
revision = model_info(model_id).sha

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    revision=revision,
)

start = time.perf_counter()

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    revision=revision,
    dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)

model.eval()
torch.cuda.synchronize()
load_seconds = time.perf_counter() - start

print("Loaded:", model_id)
print("Revision:", revision)
print("Load time:", round(load_seconds, 2), "seconds")
print("Parameter devices:", sorted({
    str(parameter.device)
    for parameter in model.parameters()
}))

config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/5.60k [00:00<?, ?B/s]

... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/182 [00:00<?, ?B/s]

Loaded: HuggingFaceTB/SmolLM3-3B
Revision: a07cc9a04f16550a088caea529712d1d335b0ac1
Load time: 103.52 seconds
Parameter devices: ['cuda:0']


In [4]:
def answer_question(question):
    messages = [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": (
                f"REFERENCE:\n{reference}\n\n"
                f"QUESTION:\n{question}"
            ),
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = tokenizer(text, return_tensors="pt").to(
        model.get_input_embeddings().weight.device
    )

    input_tokens = inputs["input_ids"].shape[1]

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()

    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            **generation_settings,
            pad_token_id=tokenizer.eos_token_id,
        )

    torch.cuda.synchronize()
    seconds = time.perf_counter() - start
    peak_memory = torch.cuda.max_memory_allocated() / 1024**3

    output_ids = generated[0, input_tokens:]
    response = tokenizer.decode(
        output_ids,
        skip_special_tokens=True,
    ).strip()

    result = {
        "response": response,
        "seconds": round(seconds, 4),
        "input_tokens": input_tokens,
        "output_tokens": len(output_ids),
        "peak_allocated_gpu_gib": round(peak_memory, 3),
        "hit_token_limit": (
            len(output_ids) >= generation_settings["max_new_tokens"]
        ),
    }

    del inputs, generated
    return result

## Pilot test

One calculation question checks generation before the main batch.
The pilot is saved separately and excluded from the final comparison.

In [5]:
packages = {
    name: importlib.metadata.version(name)
    for name in [
        "torch", "transformers", "accelerate",
        "safetensors", "huggingface_hub",
    ]
}

device_map = getattr(model, "hf_device_map", {}) or {}

run_config = {
    "model": model_id,
    "revision": revision,
    "precision": "float16",
    "thinking_enabled": False,
    "system_prompt": system_prompt,
    "generation_settings": generation_settings,
    "dataset_sha256": fingerprint,
    "python": platform.python_version(),
    "packages": packages,
    "gpu": torch.cuda.get_device_name(0),
    "device_map": {
        key: str(value) for key, value in device_map.items()
    },
    "parameter_devices": sorted({
        str(parameter.device)
        for parameter in model.parameters()
    }),
    "load_seconds": round(load_seconds, 2),
    "timing_scope": (
        "model.generate only; excludes prompt formatting, "
        "tokenization, decoding and file saving"
    ),
}

(project / "results" / "smollm_config.json").write_text(
    json.dumps(run_config, indent=2)
)

pilot_question = "What is the total for two waters and one chips?"
pilot = answer_question(pilot_question)

(project / "results" / "smollm_pilot.json").write_text(
    json.dumps({
        "question": pilot_question,
        "stage": "pilot; excluded from main batch",
        **pilot,
    }, indent=2)
)

print("Response:", pilot["response"])
print("Generation time:", pilot["seconds"], "seconds")
print("Peak allocation:", pilot["peak_allocated_gpu_gib"], "GiB")
print("Hit token limit:", pilot["hit_token_limit"])

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Response: The total for two waters and one chips is $3.00 each for water (2 x $3.00 = $6.00) plus $4.00 for chips, totaling $6.00 + $4.00 = $10.00.

Reference: [R2]
Generation time: 7.8474 seconds
Peak allocation: 5.833 GiB
Hit token limit: False


## Final experiment

All 30 fixed questions are tested with the same reference, instructions
and generation settings used for Qwen. Results are saved after each
question. The pilot is excluded from the final batch.

In [6]:
import pandas as pd
from datetime import datetime, timezone

batch_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

output_file = project / "outputs" / f"smollm_{batch_id}.json"
metrics_file = project / "results" / f"smollm_metrics_{batch_id}.csv"

batch_config = {
    **run_config,
    "batch_id": batch_id,
    "question_ids": [case["id"] for case in test_cases],
    "stage": "final experiment",
}

(project / "results" / f"smollm_config_{batch_id}.json").write_text(
    json.dumps(batch_config, indent=2)
)

records = []

for index, case in enumerate(test_cases, start=1):
    print(
        f"Running {index}/30: {case['id']} "
        f"({case['category']})",
        flush=True,
    )

    result = answer_question(case["question"])

    records.append({
        "id": case["id"],
        "category": case["category"],
        "question": case["question"],
        "model": model_id,
        "revision": revision,
        **result,
    })

    # Save progress
    output_file.write_text(json.dumps(records, indent=2))
    pd.DataFrame(records).to_csv(metrics_file, index=False)

results = pd.DataFrame(records)

assert len(results) == 30
assert not results["id"].duplicated().any()
assert set(results["id"]) == {case["id"] for case in test_cases}

print("\nFinished:", len(results), "responses")
print(
    "Mean generation time:",
    round(results["seconds"].mean(), 3),
    "seconds",
)
print(
    "Peak GPU allocation:",
    results["peak_allocated_gpu_gib"].max(),
    "GiB",
)
print(
    "Responses hitting token limit:",
    int(results["hit_token_limit"].sum()),
)

display(results[["id", "category", "response", "seconds"]])

Running 1/30: Q01 (direct)
Running 2/30: Q02 (direct)
Running 3/30: Q03 (direct)
Running 4/30: Q04 (direct)
Running 5/30: Q05 (direct)
Running 6/30: Q06 (direct)
Running 7/30: Q07 (direct)
Running 8/30: Q08 (direct)
Running 9/30: Q09 (direct)
Running 10/30: Q10 (direct)
Running 11/30: Q11 (reasoning)
Running 12/30: Q12 (reasoning)
Running 13/30: Q13 (reasoning)
Running 14/30: Q14 (reasoning)
Running 15/30: Q15 (reasoning)
Running 16/30: Q16 (reasoning)
Running 17/30: Q17 (reasoning)
Running 18/30: Q18 (reasoning)
Running 19/30: Q19 (reasoning)
Running 20/30: Q20 (reasoning)
Running 21/30: Q21 (unsupported_or_misleading)
Running 22/30: Q22 (unsupported_or_misleading)
Running 23/30: Q23 (unsupported_or_misleading)
Running 24/30: Q24 (unsupported_or_misleading)
Running 25/30: Q25 (unsupported_or_misleading)
Running 26/30: Q26 (unsupported_or_misleading)
Running 27/30: Q27 (unsupported_or_misleading)
Running 28/30: Q28 (unsupported_or_misleading)
Running 29/30: Q29 (unsupported_or_misleadi

,id,category,response,seconds
0,Q01,direct,The store opens at 06:00 on Monday.\n\nReferen...,4.3541
1,Q02,direct,The store closes at 21:00 on Sunday.\n\nRefere...,3.5604
2,Q03,direct,The regular price of one water is $3.00 each.\...,2.8808
3,Q04,direct,The regular price of one coffee is $4.50 each....,2.0124
4,Q05,direct,One coffee and one chocolate cost $6.00 under ...,1.6117
5,Q06,direct,Chips: 15 units.\n\nReference: [R4],1.4377
6,Q07,direct,The reorder threshold for water is 10 units. \...,1.0033
7,Q08,direct,Deliveries are on Tuesday and Friday.\n\nRefer...,0.9202
8,Q09,direct,"You must record the item and time found, then ...",1.5506
9,Q10,direct,[Duty Manager] approves discounts outside the ...,1.4170


In [7]:
import shutil

backup = shutil.make_archive(
    "/content/pair2_both_models_backup",
    "zip",
    root_dir=str(project),
)

files.download(backup)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>